In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader, TensorDataset

In [22]:
class ChebKANLayer(nn.Module):
    def __init__(self, in_features, out_features, degree=4):
        super().__init__()
        self.in_features  = in_features
        self.out_features = out_features
        self.degree       = degree

        self.coeffs = nn.Parameter(
            torch.empty(out_features, in_features, degree + 1)
        )
        nn.init.normal_(self.coeffs, mean=0.0, std=1 / (in_features * (degree + 1)))

        # layer norm for stability
        self.norm = nn.LayerNorm(out_features)

    def chebyshev_basis(self, x):
        T = [torch.ones_like(x), x]
        for k in range(2, self.degree + 1):
            Tk = 2.0 * x * T[-1] - T[-2]
            T.append(Tk)
        return torch.stack(T, dim=-1)

    def forward(self, x):
        x_mapped = torch.tanh(x)
        T        = self.chebyshev_basis(x_mapped)
        out      = torch.einsum('bid,oid->bo', T, self.coeffs)
        out      = self.norm(out)
        return out

In [23]:
class ChebKANViaSHAP(nn.Module):
    def __init__(self, n_features, n_classes, degree=4, hidden_dims=None):
        super().__init__()
        self.n_features = n_features
        self.n_classes  = n_classes

        if hidden_dims is None:
            hidden_dims = [64, 128, 64]

        dims = [n_features] + hidden_dims  # [18, 64, 128, 64]

        # main ChebKAN layers
        self.layers = nn.ModuleList([
            ChebKANLayer(dims[i], dims[i+1], degree=degree)
            for i in range(len(dims) - 1)
        ])

        # residual projections — needed when in/out dims differ
        self.residuals = nn.ModuleList([
            nn.Linear(dims[i], dims[i+1], bias=False)
            if dims[i] != dims[i+1] else nn.Identity()
            for i in range(len(dims) - 1)
        ])

        # dropout for regularization
        self.dropout = nn.Dropout(p=0.1)

        # output head
        self.output_layer = nn.Linear(dims[-1], n_features * n_classes)

    def forward(self, x):
        h = x
        for layer, residual in zip(self.layers, self.residuals):
            h = layer(h) + residual(h)   # residual connection
            h = self.dropout(h)

        phi   = self.output_layer(h)
        phi   = phi.view(-1, self.n_features, self.n_classes)
        y_hat = torch.sigmoid(phi.sum(dim=1))
        return phi, y_hat

In [ ]:
def sample_coalition_mask(batch_size, n_features, device):
    return torch.randint(0, 2, (batch_size, n_features),
                         dtype=torch.float32, device=device)

def viashap_loss(model, x, y, beta=10.0, n_samples=32, class_weights=None):
    device     = x.device
    batch_size = x.shape[0]
    x_baseline = torch.zeros_like(x)

    # ── compute phi once from full x (outside the loop) ──────────────
    phi, y_hat_full = model(x)
    # phi shape: [batch, n_features, n_classes]

    total_phi_loss = torch.tensor(0.0, device=device)

    with torch.no_grad():                      # baselines need no grad
        _, y_base = model(x_baseline.expand_as(x))

    for _ in range(n_samples):
        S    = sample_coalition_mask(batch_size, x.shape[1], device)
        x_S  = x * S + x_baseline * (1 - S)

        _, y_S = model(x_S)

        # Shapley axiom: f(x_S) - f(baseline) ≈ sum of phi for features in S
        diff         = y_S - y_base                        # [batch, n_classes]
        phi_selected = (phi * S.unsqueeze(-1)).sum(dim=1)  # [batch, n_classes]
        phi_loss     = ((diff - phi_selected) ** 2).mean()

        total_phi_loss = total_phi_loss + phi_loss

    total_phi_loss = total_phi_loss / n_samples

    # ── Bug 3 fix: use raw logits, not sigmoid output ─────────────────
    # phi.sum(dim=1) are the raw logits before sigmoid
    logits    = phi.sum(dim=1)                 # [batch, n_classes]
    pred_loss = F.cross_entropy(
        logits, y.long(), weight=class_weights
    )

    # ── pred loss weighted 5x so classifier learns before phi takes over
    total_loss = 5.0 * pred_loss + beta * total_phi_loss
    return total_loss, pred_loss.item(), total_phi_loss.item()

In [12]:
import pandas as pd
import openml

# Download the Elevators dataset (ID 846) from OpenML
data = openml.datasets.get_dataset(846)

# Extract the data as a pandas DataFrame
df, _, _, _ = data.get_data(dataset_format="dataframe")

print("Columns:", df.columns.tolist())
print("Shape:  ", df.shape)


Columns: ['climbRate', 'Sgz', 'p', 'q', 'curRoll', 'absRoll', 'diffClb', 'diffRollRate', 'diffDiffClb', 'SaTime1', 'SaTime2', 'SaTime3', 'SaTime4', 'diffSaTime1', 'diffSaTime2', 'diffSaTime3', 'diffSaTime4', 'Sa', 'binaryClass']
Shape:   (16599, 19)


In [13]:
target_col = "binaryClass"

X = df.drop(columns=[target_col]).values.astype(np.float32)

# fix: encode P/N to 0/1
le = LabelEncoder()
y  = le.fit_transform(df[target_col].values).astype(np.int64)

print("Label mapping:", dict(zip(le.classes_, le.transform(le.classes_))))
# should print: {'N': 0, 'P': 1}  or {'P': 0, 'N': 1}

scaler = StandardScaler()
X      = scaler.fit_transform(X)

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, 
                                                     random_state=42, 
                                                     stratify=y)   # ← add stratify because imbalanced
X_val,   X_test, y_val,   y_test = train_test_split(X_temp, y_temp, test_size=0.50, 
                                                     random_state=42,
                                                     stratify=y_temp)  # ← same here

print(f"Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}")
print(f"Train class balance: {np.bincount(y_train)}")
print(f"Val   class balance: {np.bincount(y_val)}")

to_tensor    = lambda a: torch.tensor(a)
train_loader = DataLoader(TensorDataset(to_tensor(X_train), to_tensor(y_train)), 
                          batch_size=128, shuffle=True)
val_loader   = DataLoader(TensorDataset(to_tensor(X_val),   to_tensor(y_val)),   
                          batch_size=512, shuffle=False)
test_loader  = DataLoader(TensorDataset(to_tensor(X_test),  to_tensor(y_test)),  
                          batch_size=512, shuffle=False)

Label mapping: {'N': 0, 'P': 1}
Train: (11619, 18) | Val: (2490, 18) | Test: (2490, 18)
Train class balance: [3591 8028]
Val   class balance: [ 770 1720]


In [24]:
device     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
n_features = X.shape[1]
n_classes  = 2

counts        = np.bincount(y_train)
class_weights = torch.tensor(
    [len(y_train) / (2 * counts[0]),
     len(y_train) / (2 * counts[1])],
    dtype=torch.float32
).to(device)

model = ChebKANViaSHAP(
    n_features  = n_features,
    n_classes   = n_classes,
    degree      = 4,
    hidden_dims = [64, 128, 64]
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5, verbose=True
)

print(f"Device:     {device}")
print(f"Features:   {n_features}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Class weights: N={class_weights[0]:.4f}, P={class_weights[1]:.4f}")

Device:     cuda
Features:   18
Parameters: 108,068
Class weights: N=1.6178, P=0.7237


c:\Users\neera\OneDrive\Desktop\xai\ViaSHAP\venv\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


In [25]:
BETA      = 10     # back to paper default
N_SAMPLES = 32
EPOCHS    = 150
PATIENCE  = 20

best_val_loss    = float("inf")
patience_counter = 0

torch.save(model.state_dict(), "chebkan_viashap_best.pt")

for epoch in range(1, EPOCHS + 1):

    # ── Train ──────────────────────────────────────────
    model.train()
    epoch_loss  = 0.0
    last_pred_l = 0.0
    last_phi_l  = 0.0

    for x_batch, y_batch in train_loader:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()

        loss, pred_l, phi_l = viashap_loss(
            model, x_batch, y_batch,
            beta          = BETA,
            n_samples     = N_SAMPLES,
            class_weights = class_weights
        )

        loss.backward()

        # clip gradients to prevent instability
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        epoch_loss  += loss.item()
        last_pred_l  = pred_l
        last_phi_l   = phi_l

    # ── Validate ───────────────────────────────────────
    model.eval()
    val_loss              = 0.0
    val_preds, val_labels = [], []

    with torch.no_grad():
        for x_v, y_v in val_loader:
            x_v, y_v = x_v.to(device), y_v.to(device)

            vl, _, _ = viashap_loss(
                model, x_v, y_v,
                beta          = BETA,
                n_samples     = 32,
                class_weights = class_weights
            )
            val_loss += vl.item()

            _, y_hat = model(x_v)
            probs    = torch.softmax(y_hat, dim=1)[:, 1]
            val_preds.append(probs.cpu().numpy())
            val_labels.append(y_v.cpu().numpy())

    val_auc   = roc_auc_score(
        np.concatenate(val_labels),
        np.concatenate(val_preds)
    )

    avg_train = epoch_loss / len(train_loader)
    avg_val   = val_loss   / len(val_loader)

    # step scheduler based on val loss
    scheduler.step(avg_val)

    print(f"Epoch {epoch:3d} | "
          f"train={avg_train:.4f} | "
          f"val={avg_val:.4f} | "
          f"val_auc={val_auc:.4f} | "
          f"pred={last_pred_l:.4f} | "
          f"phi={last_phi_l:.6f}")

    # ── Early stopping ─────────────────────────────────
    if avg_val < best_val_loss:
        best_val_loss    = avg_val
        patience_counter = 0
        torch.save(model.state_dict(), "chebkan_viashap_best.pt")
        print(f"           ✓ saved best model (val={avg_val:.4f})")
    else:
        patience_counter += 1
        print(f"           patience {patience_counter}/{PATIENCE}")
        if patience_counter >= PATIENCE:
            print(f"Early stopping at epoch {epoch}")
            break

print(f"\nTraining complete. Best val loss: {best_val_loss:.4f}")

Epoch   1 | train=22.1857 | val=0.7397 | val_auc=0.8637 | pred=0.6863 | phi=0.008672
           ✓ saved best model (val=0.7397)
Epoch   2 | train=0.7496 | val=0.7049 | val_auc=0.9053 | pred=0.6829 | phi=0.002619
           ✓ saved best model (val=0.7049)
Epoch   3 | train=0.7206 | val=0.6870 | val_auc=0.8845 | pred=0.6794 | phi=0.002213
           ✓ saved best model (val=0.6870)
Epoch   4 | train=0.7080 | val=0.6829 | val_auc=0.8800 | pred=0.6735 | phi=0.001988
           ✓ saved best model (val=0.6829)
Epoch   5 | train=0.6950 | val=0.6818 | val_auc=0.8527 | pred=0.6683 | phi=0.002412
           ✓ saved best model (val=0.6818)
Epoch   6 | train=0.6942 | val=0.6764 | val_auc=0.8611 | pred=0.6697 | phi=0.001840
           ✓ saved best model (val=0.6764)
Epoch   7 | train=0.6858 | val=0.6765 | val_auc=0.8650 | pred=0.6570 | phi=0.002156
           patience 1/20
Epoch   8 | train=0.6843 | val=0.6707 | val_auc=0.8563 | pred=0.6550 | phi=0.002147
           ✓ saved best model (val=0.6707)
E

KeyboardInterrupt: 

In [21]:
# check best saved model so far
model.load_state_dict(torch.load("chebkan_viashap_best.pt"))
print("Best val loss so far:", best_val_loss)

Best val loss so far: 0.6516735911369324


C:\Users\neera\AppData\Local\Temp\ipykernel_50008\51353748.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("chebkan_viashap_best.pt"))


In [ ]:
model.load_state_dict(torch.load("chebkan_viashap_best.pt"))
model.eval()

all_preds, all_labels = [], []

with torch.no_grad():
    for x_t, y_t in test_loader:
        x_t       = x_t.to(device)
        _, y_hat  = model(x_t)
        probs     = torch.softmax(y_hat, dim=1)[:, 1]
        all_preds.append(probs.cpu().numpy())
        all_labels.append(y_t.numpy())

test_auc = roc_auc_score(np.concatenate(all_labels),
                          np.concatenate(all_preds))

print(f"Test AUC (ChebKAN-ViaSHAP) : {test_auc:.4f}")
print(f"Paper AUC (KANVia Elevators): 0.935")
print(f"Difference                  : {test_auc - 0.935:+.4f}")

Test AUC (ChebKAN-ViaSHAP) : 0.8522
Paper AUC (KANVia Elevators): 0.935
Difference                  : -0.0828
